# Stage 2 Notebook 29 - Exp2X Hybrid + ALL working signals combined

**Why this exists.** Companion to Exp2W (NB28). Same mask-supervision hypothesis, but applied to the architecture that has shown the most promise (hybrid prior+query, Exp2Q with decoded_f1=0.039 -- the project's highest), with two additional fixes layered on:

1. **`mask_aux: true, w_mask: 2.0`** -- the dense lane signal we've been missing across every query-style experiment.
2. **`stage1_aux_loss_weight: 0.5`** -- Exp2T's diagnosed fix for the hybrid head's missing stage 1 supervision. Forces the 192-prior generator to actually train as the geometry champion.
3. **`match_cost_iou: 4.0`** (raised from 2.0) -- when stage 1's matched_iou is good, the K=12 query selection cost should weight LineIoU more heavily so queries pick the best-IoU priors.

This combines every working ingredient identified across 22 prior experiments: prior-based geometry generator (Exp2N), query refiner (Exp2P), stage 1 supervision (Exp2T diagnosis), and dense mask supervision (CLRNet standard, restored in Exp2W).

Independent of Exp2W: Exp2W tests mask alone on the simpler query head; Exp2X tests mask plus the full set of fixes on the hybrid architecture.

Reference: CLRNet (CVPR 2022), CLRKDNet (TIP 2024), Sparse R-CNN (CVPR 2021), Mask2Former (CVPR 2022).

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 10-epoch short run.
3. Output mirrored to notebook cell, Colab runtime log, Drive log file.
4. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp24_rmt_gca_hybrid_combined_signals_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp24_rmt_gca_hybrid_combined_signals_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp24_rmt_gca_hybrid_combined_signals_joint_smoke.log
OK exp24_rmt_gca_hybrid_combined_signals_joint.yaml
  lane_shape=(1, 12, 72, 2) det_shape=(1, 4, 4)
  lane_loss=4.8741 det_loss=3.0273 grad_cos=0.0415 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.4955730438232422, 'gate/lane_mean': 0.5004420876502991, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp24_rmt_gca_hybrid_combined_signals_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp24_rmt_gca_hybrid_combined_signals_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp24_rmt_gca_hybrid_combined_signals_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp24_rmt_gca_hybrid_combined_signals_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp24_rmt_gca_hybrid_combined_signals_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp24_rmt_gca_hybrid_combined_signals_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp24_rmt_gca_hybrid_combined_signals_joint.yaml --curve-tar /con

0

## What to watch in Exp2X training

Pass criteria at epoch 10:
- **`val/lane/decoded_f1 >= 0.15`** (4-6x Exp2Q's 0.039). Combining all working signals should make a real dent.
- **`val/matched_line_iou >= 0.35`** (closer to Exp2N's 0.42, lifted by mask aux + stage 1 supervision).
- **`val/lane/decoded_oracle_f1 >= 0.20`**.
- **`val_lane_f1 >= 0.55`**.
- **`lane/stage1_aux_total`** decreases over training (stage 1 supervision is active).
- **`val/lane/mask_aux`** decreases over training (mask supervision is active).

Failure signals:
- decoded_f1 < 0.07: combining everything didn't break the impasse; the bottleneck is fundamental (training duration, resolution, KD from teacher).
- Geometry collapses: too many loss terms competing; reduce w_mask or stage1_aux_loss_weight.
- val_lane_f1 regresses: hybrid + mask conflict; revert to Exp2W.